# 牛津 Tutorial LLM 仿真 · Capstone Phase 5: 商业模式与估值

## Persona Prompt (Oxford Tutorial Fellow + HBS Devil's Advocate)

You are an **Oxford tutorial fellow** in *Capstone Phase 5: Business Model & Valuation*. Your job is one-on-one Socratic tutoring, **NEVER give direct answers**. You push the student to derive ATE->ARPU->NPV themselves, defend their Monte Carlo design, and justify their Bull/Base/Bear推演.

Rules:
1. **Socratic questioning only** - end EVERY turn with a probing question.
2. **HBS devil's advocate** - attack vague claims ("NPV looks good" is rejected; demand a number with CI).
3. **Reject vague claims** - if the student says "推理成本很重要" without a % or a tornado chart rank, push back.
4. **No direct answers** - if the student asks "What is NPV?", reply with "What would `npf.npv` need as inputs? Derive it from Phase 4 ATE."
5. **Reference real artifacts** - demand the student cite their `npf.npv(0.10, FCF)` output, P(NPV>0) number, tornado chart ranking, Bull/Base/Bear table.

本单元限频: **每单元 1 次/天** (防依赖, 鼓励先独立检索)。本仿真为静态 if/else 模拟, 不调 openai/anthropic API。

## Pre-Tutorial Task (Forced Retrieval · 必须先提交)

> 提交本段后才进入 Socratic loop。不提交 = tutorial 不开始。

**提交一段 300 字方案** (Markdown 文本即可, 不跑代码):

> 给定 Phase 4 输出 ATE=+3.8pp (95% CI: [2.2pp, 5.4pp]), 月触达 10,000, AOV $158, α=3.33%, 折现率 10%。
> 请回答:
> 1. 你如何从 ATE 推到年 ARPU? (写出推导链)
> 2. 你会如何用 `scipy.stats` 传播 ATE 的 CI 到 NPV 分布? P(NPV>0) 你预期大致多少?
> 3. 你的 Bull/Base/Bear 三路径如何与 ATE 的 CI 对齐?

把答案写入下方 `student_pre_submit` 字符串。

In [ ]:
# ====== 牛津 Tutorial Socratic Loop (静态 if/else 模拟, 不调 LLM API) ======
# 4 轮 Socratic 追问, 每轮含 >=1 个苏格拉底问 (为什么/反例/若前提变/凭什么/如何)
# 学生把 pre-tutorial 答案写入 student_pre_submit, 然后顺序运行 4 轮

student_pre_submit = """
ATE=3.8pp, 月触达 10000, AOV=158, alpha=3.33%。
ARPU = 10000 * 0.038 * 158 * 0.0333 * 12 ≈ 24000/yr。
蒙特卡洛: 用 normal 抽 ATE, N=10000 次, 算 P(NPV>0)。
Bull=ATE=5.4pp, Base=3.8pp, Bear=2.2pp。
"""

turns = []

# ---- Turn 1: 攻击 "normal 抽 ATE" 的方法论 ----
turns.append({
    "round": 1,
    "fellow": (
        "你写 '用 normal 抽 ATE'。**凭什么用 normal 而非 truncnorm?** "
        "ATE 的 95% CI 是 [2.2pp, 5.4pp], normal 抽样会产出负 ATE - 转化率提升为负 "
        "在 Phase 4 已观测 +3.8pp 的前提下合理吗? **请给出反例: 若 normal 抽到 ATE=-1pp, "
        "你的 ARPU 会变成多少? 这是否违背 Phase 4 已观测数据?**"
    ),
    "expected_student_response": "应改用 scipy.stats.truncnorm, 截断在 [0, +inf) 或 [0.022, 0.054] CI 内; normal 抽到 -1pp 会让 ARPU 变负, 违背 Phase 4 已观测 +3.8pp。"
})
# 苏格拉底问 #1 (凭什么), #2 (反例) - 已含

# ---- Turn 2: 攻击 ARPU 推导链的 α 来源 ----
turns.append({
    "round": 2,
    "fellow": (
        "你的 ARPU 推导链用 α=3.33%。**这个 3.33% 从哪来? 为什么不是 5% 或 1%?** "
        "若 α 翻倍到 6.66%, 你的 NPV 会怎样变化? **如何用龙卷风图量化 α 对 NPV 的杠杆?** "
        "请引用你 practice.md D3 的龙卷风图排序逻辑。"
    ),
    "expected_student_response": "α=3.33% 来自技能4 Day 2 outcome-based pricing 价值捕获率; α 翻倍 -> ARPU 翻倍 -> FCF 翻倍 -> NPV 显著上升; 龙卷风图按 |ΔNPV| 降序排, α 应排进前 3。"
})
# 苏格拉底问 #3 (如何), #4 (为什么) - 已含

# ---- Turn 3: 攻击 Bull/Base/Bear 与 CI 对齐 ----
turns.append({
    "round": 3,
    "fellow": (
        "你写 Bull=5.4pp / Base=3.8pp / Bear=2.2pp, 与 ATE CI 对齐。但 **天道推演要求每路径推演 "
        "immediate/near/far 三层。你的 Bear 路径在 far 层 (3年后) 会发生什么?** "
        "若 Bear 路径下推理成本因 DeepSeek 效应再降 90%, **far 层的毛利率与 NPV 是否会被 '救回'?** "
        "**反例: 若竞争对手同期也用 DeepSeek, Bear 路径的 NPV 还能救回吗?**"
    ),
    "expected_student_response": "Bear far 层: ATE=2.2pp 导致 ARPU 低, 但 DeepSeek -90% 推理成本可拉高毛利率, 部分对冲; 但若竞争对手也降本, 价格战压 AOV, NPV 仍可能为负。"
})
# 苏格拉底问 #5 (反例/若前提变) - 已含

# ---- Turn 4: 攻击 P(NPV>0) 的决策解释 ----
turns.append({
    "round": 4,
    "fellow": (
        "假设你的蒙特卡洛输出 P(NPV>0)=0.72。**这个 0.72 够投还是不够投? 决策阈值是多少?** "
        "**为什么不是 0.5 或 0.95?** 你的阈值是否应随 AI 项目的 J 曲线效应调整? "
        "**若贝叶斯估值 (PyMC) 给出 P(NPV>0 | Phase 4 ATE) = 0.85, 你会更信哪个数? 为什么?**"
    ),
    "expected_student_response": "0.72 在多数风投阈值 (>=0.7) 边缘; J 曲线前期亏损大, 阈值应更高 (如 0.8); 贝叶斯 0.85 更可信因用先验正则化 + 可随新数据更新, 但需警惕先验主观性。"
})
# 苏格拉底问 #6 (为什么/凭什么) - 已含 (超过 5 个)

# 静态 if/else 模拟: 检查 student_pre_submit 是否含关键元素, 给不同反馈
print("=" * 70)
print("牛津 Tutorial Socratic Loop (4 轮, 静态模拟)")
print("=" * 70)
for t in turns:
    print(f"\n[Round {t['round']}] Fellow (Socratic):")
    print(t["fellow"])
    print(f"\n[Expected student response]:")
    print(t["expected_student_response"])

# 静态判断学生提交质量
print("\n" + "=" * 70)
print("Tutorial Fellow 静态诊断:")
print("=" * 70)
if "truncnorm" not in student_pre_submit and "normal" in student_pre_submit.lower():
    print("- 盲点 #1: 用 normal 抽 ATE, 应改 truncnorm (Round 1 已追问)")
if "3.33" not in student_pre_submit and "alpha" not in student_pre_submit.lower():
    print("- 盲点 #2: 未说明 α 来源 (Round 2 已追问)")
if "immediate" not in student_pre_submit.lower() and "near" not in student_pre_submit.lower():
    print("- 盲点 #3: 未写三路径三层推演 (Round 3 已追问)")
if "0.72" not in student_pre_submit and "P(NPV>0)" not in student_pre_submit:
    print("- 盲点 #4: 未给 P(NPV>0) 决策阈值 (Round 4 已追问)")
print("\n=> 共 4 轮 Socratic 追问, 含 6 个苏格拉底问 (为什么/凭什么/反例/若前提变/如何)。")


In [ ]:
# ====== student_model.json: 记录掌握度/盲点 (供下次 tutorial 优先追问) ======
import json, os

student_model = {
    "unit": "capstone-phase-5",
    "student_id": "demo-student",
    "mastery": {
        "S1_DCF_npf": 0.6,          # D1: ATE->ARPU->NPV, subskill A
        "S2_MonteCarlo_truncnorm": 0.3,  # D2: 蒙特卡洛, subskill B (盲点: 用 normal 而非 truncnorm)
        "S3_Tornado_TianDao": 0.5   # D3: 龙卷风图+三路径, subskill C
    },
    "blindspots": [
        "用 normal 抽 ATE (应 truncnorm, 见 Round 1)",
        "未说明 α=3.33% 来源 (Round 2)",
        "Bear far 层未考虑 DeepSeek 对冲 + 竞争对手反应 (Round 3)",
        "P(NPV>0) 决策阈值未与 J 曲线挂钩 (Round 4)"
    ],
    "weak_subskill": "S2_MonteCarlo_truncnorm",  # 触发 practice.md weak_loop
    "next_tutorial_priority": ["S2", "S3"],
    "sessions_today": 1,
    "daily_limit": 1   # 限频: 每单元 1 次/天
}

out_path = "student_model.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)
print(f"Wrote {out_path} (size={os.path.getsize(out_path)}B)")
print(json.dumps(student_model, ensure_ascii=False, indent=2))

# 读回校验
with open(out_path, "r", encoding="utf-8") as f:
    reloaded = json.load(f)
assert reloaded["blindspots"] == student_model["blindspots"], "student_model read/write mismatch"
print("\n[OK] student_model.json 读写一致。下次 tutorial 将优先追问:", reloaded["next_tutorial_priority"])


## Hattie 四级形成性反馈 (Visible Learning, 2009)

> Hattie & Timperley (2007) 四级反馈框架。避免 Self 级表扬 (Hattie: 表扬效应量 d=0.12, 几乎无效)。

**[TASK] 任务级反馈** (关于本次 tutorial 的具体任务对错):
- 你的 ARPU 推导链数学正确 (10000×0.038×158×0.0333×12 ≈ $24K), 但 α=3.33% 来源未交代。
- 蒙特卡洛用 normal 是错的 (会抽到负 ATE), 应改 `scipy.stats.truncnorm`。

**[PROCESS] 过程级反馈** (关于解题策略/元认知):
- 你直接假设 ARPU 而非从 ATE 推导 - 缺少 Phase 4->Phase 5 的因果闭环。
- Bull/Base/Bear 只写了 ATE 数值, 未推演 immediate/near/far 三层 - 天道推演的沙盘深度不够。
- 建议: 每路径先用一句话写 immediate (1年内), 再 near (1-3年), 再 far (3年+), 检查路径间分歧点。

**[SELF-REG] 自我调节级反馈** (关于学生自我监控/调整):
- 你在 Round 1 被追问后能否自己发现 truncnorm 的需要? 若不能, 说明 S2 子技能的元认知监控弱。
- 建议: 每次写蒙特卡洛前自问 "这个分布会抽到不合物理意义的值吗?" - 这是可迁移的自检习惯。

**[FEED-FORWARD] 前馈级反馈** (关于下一步该做什么):
- 下次 tutorial (限频: 1次/天, 明天再来) 优先追问 S2 (蒙特卡洛) + S3 (三路径)。
- 今晚推荐: (1) 重做 practice.md D2 Stage A (worked); (2) 读 solution.ipynb TODO4 看正确 truncnorm 调用; (3) 用 PyMC 选做贝叶斯估值, 对比频率派 P(NPV>0)。

> 注: 故意 **不写 Self 级表扬** (如 "你做得很好")。Hattie 元分析显示表扬效应量极低, 反而可能削弱内在动机。

## 限频与退出产物 (防依赖 + 退出盲点)

### 限频 (Rate Limit)
- **每单元 1 次/天**: 本 tutorial 仿真每天最多跑 1 轮 (4 Socratic turns)。`student_model.json` 的 `sessions_today` 字段记录。
- **为何限频**: 防止学生依赖 LLM 仿真而非独立检索。Hattie 反馈研究: 过度依赖外部脚手架会削弱 retrieval practice 效应 (Karpicke & Roediger 2008)。
- **解除**: 明天 0 点重置 `sessions_today=0`, 可再跑一轮。

### 退出产物 (Exit Artifact)
跑完本 tutorial 后, 在 `student_model.json` 的 `exit_artifact` 字段写入:

1. **2-3 个盲点** (从 Round 1-4 的 Socratic 追问中提炼):
   - 例: "我用 normal 抽 ATE, 没考虑负值不合物理意义"
   - 例: "我 Bull/Base/Bear 只写 ATE 数值, 没推演 far 层的竞争对手反应"
   - 例: "我没把 P(NPV>0) 阈值与 AI 项目 J 曲线挂钩"

2. **推荐复习单元** (跨单元回溯):
   - Phase 4 (因果推断): 复习 ATE 的定义与 CI 解读, 理解为何 truncnorm 截断在 [0.022, 0.054]。
   - 技能4 Day 2 (AI 定价策略): 复习 outcome-based pricing 与 α=3.33% 的来源。
   - 技能4 Day 3 (Agent 经济学): 复习推理成本对毛利率的影响, 理解 DeepSeek 效应。

3. **下次 tutorial 自定义问题**: 写 1 个你想被追问的问题 (训练 self-questioning)。

---

### 引用
- Hattie, J. & Timperley, H. (2007). The Power of Feedback. *Review of Educational Research*.
- Karpicke, J. & Roediger, H. (2008). The Critical Importance of Retrieval for Learning. *Science*.
- Biggs, J. (1996). Enhancing teaching through constructive alignment.
- 本单元 notes.md § 关键回顾 2 (ATE->ARPU->NPV) + § 2026前沿 (贝叶斯估值/推理成本/天道推演)